In [ ]:
# basic notebook for plotting logos from annotated seqlets for highlighting specific examples #

In [1]:
# import packages
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import torch
# tangermeme packages
from tangermeme.seqlet import recursive_seqlets
from tangermeme.plot import plot_logo
from tangermeme.annotate import annotate_seqlets
from tangermeme.io import read_meme
import logomaker

In [2]:
# define a function for reading in data and concatenating #
def read_in_sat_mut (path2satmut, chunksize):
    # define a list for concatenating #
    chunks2cat = []
    # iterate through chunked tsv #
    for chunk in tqdm(pd.read_csv(path2satmut, sep = '\t', chunksize=chunksize)):
        chunks2cat.append(chunk)
    # concatenate chunks #
    cat_df = pd.concat(chunks2cat)
    return cat_df

In [3]:
# define function for reformatting collapsed mpac seqlet calls:
def mpac_collapsed_dels (path2bed,
                    motif_dict):
    # open bed file
    bed = pd.read_csv(path2bed, sep = '\t', header = None)
    filtered_bed = bed.copy()
    # Use column 5 (rep_tf) which has the TF with highest |contribution| for multi-TF intervals
    # Try multiple HOCOMOCO suffixes since they vary (.A, .B, .C, .D)
    def get_tf_family(rep_tf, motif_dict):
        for suffix in ['.A', '.B', '.C', '.D']:
            key = f"{rep_tf}_HUMAN.H11MO.0{suffix}"
            if key in motif_dict:
                return motif_dict[key]
        return None
    # reformat the bed for plotting purposes
    bed2plot = pd.DataFrame({'chrom' : filtered_bed[0],
                             'start' : filtered_bed[1],
                             'end' : filtered_bed[2],
                             'tf' : filtered_bed[5],  # Use pre-computed rep_tf from column 5
                             'tf_family' : [get_tf_family(tf, motif_dict) for tf in filtered_bed[5]],
                             'activity_class' : filtered_bed[6],
                             'enhancer_id' : filtered_bed[7]})
    return bed2plot

In [4]:
# open vierstra clusters for collapsing on families instead of tfs
vierstra_motifs = pd.read_excel('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/gnomad_buffering_analysis/motif_annotations.xlsx', sheet_name=[0,1])
# make a dictionary out of the clusterIDs and Names - this will be used to generate the final dictionary with the motif names
idName_dict = dict(zip(vierstra_motifs[0]['Cluster_ID'], vierstra_motifs[0]['Name']))
# open the second page and assign the Names to the individual motifs
vierstra_motifs[1].loc[:,'cluster_name'] = [idName_dict.get(i) for i in vierstra_motifs[1]['Cluster_ID']]
# make a dictionary that pulls the motif name as a key and returns the vierstra family as a value
vierstra_motif_dict = dict(zip(vierstra_motifs[1]['Motif'], vierstra_motifs[1]['cluster_name']))

In [5]:
# reformat MPAC collapsed BED files
path2beds = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/bed_files'
# k562
k_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_k562_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)
# hepg2
h_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_hepg2_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)
# sknsh
s_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_sknsh_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)

In [6]:
# open data from tian
tian_raw_data = pd.read_csv('gtex_meta_result_with_global_eqtl_stats.tsv', sep = '\t')
# open dELS bed file
dELS_bed = pd.read_csv('../../../processed_data/GRCh38-dELS-only.bed', sep = '\t', header = None)
# open the variant data with the raw gtex info
varsWithGTEx = pd.read_csv('all_highPIP_with_GTEx_Tissue_Data.tsv', sep = '\t')

In [7]:
varsWithGTEx.head()

,phenotype_id,gene_name,biotype,variant_id,pip,af,cs_id,cs_size,afc,afc_se,tissue,tss_distance,pval_nominal,slope,slope_se,pval_nominal_threshold,min_pval_nominal,pval_beta,rna_samples,rna_count
0,ENSG00000230699.2,ENSG00000230699,lncRNA,chr1_906982_C_T_b38,1.000000,0.245928,1,1,-0.732348,0.141169,Esophagus_Mucosa,-4453,3.884256765385854e-16,-0.47403568029403687,0.056428469717502594,0.000222799,3.88426e-16,7.95744e-12,NaN,NaN
1,ENSG00000157916.20,RER1,protein_coding,chr1_2397275_A_G_b38,0.981109,0.882736,1,1,0.169547,0.055328,Esophagus_Mucosa,5500,2.2806442552865136e-18,0.2215828001499176,0.02444879338145256,0.000102268,2.28064e-18,5.37739e-14,NaN,NaN
2,ENSG00000127481.15,UBR4,protein_coding,chr1_19110691_G_T_b38,0.926916,0.169381,1,2,0.262016,0.075773,Esophagus_Mucosa,-99575,5.0399563744727854e-27,0.2267777919769287,0.019944937899708748,0.000130823,5.03997e-27,3.46562e-22,NaN,NaN
3,ENSG00000085998.15,POMGNT1,protein_coding,chr1_46183374_C_T_b38,0.957260,0.144951,1,1,0.166659,0.048644,Esophagus_Mucosa,-19864,4.2235993652650593e-10,0.17711658775806427,0.0278389323502779,0.000408085,4.2236e-10,1.59626e-06,NaN,NaN
4,ENSG00000171517.6,LPAR3,protein_coding,chr1_84888880_G_A_b38,0.951204,0.300489,1,1,-0.340618,0.070038,Esophagus_Mucosa,-4326,1.3415322597448859e-16,-0.2672632336616516,0.03129248693585396,0.000209033,1.34153e-16,3.1801e-12,NaN,NaN


In [8]:
# merge all above data together for batch processing
# add an id column to the gtex_data
varsWithGTEx.loc[:, 'merge_id'] = [(':').join([varID, tissue, gene]) for varID, tissue, gene in zip(varsWithGTEx['variant_id'], varsWithGTEx['tissue'], varsWithGTEx['gene_name'])]
# reformat variant id and make matching id in Tian's df
tian_raw_data.loc[:, 'gtex_id'] = [('_').join(i.split(':')) + '_b38' for i in tian_raw_data['Variant_ID']]
tian_raw_data.loc[:, 'gtex_id_leadVar'] = [('_').join(i.split(':')) + '_b38' for i in tian_raw_data['Leadvar']]
# build merge id
tian_raw_data.loc[:, 'merge_id'] = [(':').join([gtexID, tissue, gene]) for gtexID, tissue, gene in zip(tian_raw_data['gtex_id'], tian_raw_data['Tissue_Name'], tian_raw_data['Gene_Name'])]
# make some dictionaries for adding data to Tian's df
enhStart = dict(zip(dELS_bed[4], dELS_bed[1]))
varPIP = dict(zip(varsWithGTEx['merge_id'], varsWithGTEx['pip']))
leadPvalNom = dict(zip(varsWithGTEx['merge_id'], varsWithGTEx['pval_nominal']))
leadPvalBeta = dict(zip(varsWithGTEx['merge_id'], varsWithGTEx['pval_beta']))
tssDist = dict(zip(varsWithGTEx['merge_id'], varsWithGTEx['tss_distance']))

In [9]:
# add those data to tian's dataframe
# enhancer start
tian_raw_data.loc[:, 'enhancer_start'] = [enhStart.get(i) if i in enhStart.keys() else '' for i in tian_raw_data['enhancer_ids']]
tian_raw_data.loc[:, 'lead_pip'] = [varPIP.get(i) if i in varPIP.keys() else '' for i in tian_raw_data['merge_id']]
tian_raw_data.loc[:, 'lead_pval_nom'] = [leadPvalNom.get(i) if i in leadPvalNom.keys() else '' for i in tian_raw_data['merge_id']]
tian_raw_data.loc[:, 'lead_pval_beta'] = [leadPvalBeta.get(i) if i in leadPvalBeta.keys() else '' for i in tian_raw_data['merge_id']]
tian_raw_data.loc[:, 'tss_dist'] = [tssDist.get(i) if i in tssDist.keys() else '' for i in tian_raw_data['merge_id']]

In [10]:
# open plotted points from Tian
plotPoints = pd.read_csv('plotted_windows_metadata_v10_drop40leadvar.tsv', sep = '\t')
# add GTEx ID
plotPoints.loc[:, 'gtex_id_leadVar'] = [('_').join(i.split(':')) + '_b38' for i in plotPoints['Leadvar']]

In [11]:
# filter for those plotted points to speed up plotting/make sense of the important examples
# build new id for matching to the plot points like so, gtex_id:tissue:cell_type
plotPoints.loc[:, 'plotFilter_id'] = [(':').join([gtex_id, tissue, cell]) for gtex_id, tissue, cell in zip(plotPoints['gtex_id_leadVar'], plotPoints['Tissue'], plotPoints['Celltype'])]
# check that id is unique
print(f'the number of unique ids matches the length of the df: {len(plotPoints) == len(plotPoints['plotFilter_id'].unique())}')
# make the same id in the raw data df
tian_raw_data.loc[:, 'plotFilter_id'] = [(':').join([gtex_id, tissue, cell]) for gtex_id, tissue, cell in zip(tian_raw_data['gtex_id_leadVar'], tian_raw_data['Tissue_Name'], tian_raw_data['cell_type'])]
# add gene to another filter id
tian_raw_data.loc[:, 'fullFilter_id'] = [(':').join([plotID, gene]) for plotID, gene in zip(tian_raw_data['plotFilter_id'], tian_raw_data['Gene_Name'])]

the number of unique ids matches the length of the df: True


In [12]:
plotPoints.head()

,Leadvar,Tissue,Celltype,z_lead,z_rare,n_rare,is_phenocopy,gtex_id_leadVar,plotFilter_id
0,chr10:46796436:C:T,Esophagus_Gastroesophageal_Junction,hepg2,1.469709,-0.134122,10,False,chr10_46796436_C_T_b38,chr10_46796436_C_T_b38:Esophagus_Gastroesophag...
1,chr10:46796436:C:T,Esophagus_Muscularis,hepg2,1.525284,-0.216560,16,False,chr10_46796436_C_T_b38,chr10_46796436_C_T_b38:Esophagus_Muscularis:hepg2
2,chr10:46796436:C:T,Pancreas,hepg2,1.362827,-0.493983,11,False,chr10_46796436_C_T_b38,chr10_46796436_C_T_b38:Pancreas:hepg2
3,chr10:6053965:C:A,Adipose_Visceral_Omentum,hepg2,-0.173763,-0.559323,1,True,chr10_6053965_C_A_b38,chr10_6053965_C_A_b38:Adipose_Visceral_Omentum...
4,chr10:6053965:C:A,Colon_Sigmoid,hepg2,-0.183048,1.969365,1,False,chr10_6053965_C_A_b38,chr10_6053965_C_A_b38:Colon_Sigmoid:hepg2


In [13]:
# make some dictionaries for adding the z-scores and 'is_phenocopy' annotation to df to pass to plotting function
zLeadDict = dict(zip(
    plotPoints['plotFilter_id'],
    plotPoints['z_lead']
))
zRareDict = dict(zip(
    plotPoints['plotFilter_id'],
    plotPoints['z_rare']
))
phenocopyDict = dict(zip(
    plotPoints['plotFilter_id'],
    plotPoints['is_phenocopy']
))

In [14]:
# filter the full raw data for only those plotted examples
tian_plotted_data = tian_raw_data[tian_raw_data['plotFilter_id'].isin(plotPoints['plotFilter_id'].tolist())].copy()
# add the z-scores
# lead
tian_plotted_data.loc[:, 'z_lead'] = [zLeadDict.get(i) for i in tian_plotted_data['plotFilter_id']]
# rare
tian_plotted_data.loc[:, 'z_rare'] = [zRareDict.get(i) for i in tian_plotted_data['plotFilter_id']]
# phenocopy bool
tian_plotted_data.loc[:, 'is_phenocopy'] = [phenocopyDict.get(i) for i in tian_plotted_data['plotFilter_id']]

In [15]:
tian_plotted_data.head()

,Variant_ID,Leadvar,Variant_Tag,Subject_ID,Tissue_Abbr,Tissue_Name,Genotype(ref0/alt1),TPM,skew_pred,cell_type,...,enhancer_start,lead_pip,lead_pval_nom,lead_pval_beta,tss_dist,plotFilter_id,fullFilter_id,z_lead,z_rare,is_phenocopy
30724,chr1:1439425:G:C,chr1:1439454:A:G,rare,GTEX-111CU,Stomac,Stomach,0,0.5303,-0.726346,k562,...,1439414,,,,,chr1_1439454_A_G_b38:Stomach:k562,chr1_1439454_A_G_b38:Stomach:k562:ATAD3C,-0.168652,0.159428,False
30725,chr1:1439425:G:C,chr1:1439454:A:G,rare,GTEX-111YS,Stomac,Stomach,0,0.2302,-0.726346,k562,...,1439414,,,,,chr1_1439454_A_G_b38:Stomach:k562,chr1_1439454_A_G_b38:Stomach:k562:ATAD3C,-0.168652,0.159428,False
30726,chr1:1439425:G:C,chr1:1439454:A:G,rare,GTEX-1122O,Stomac,Stomach,0,0.1718,-0.726346,k562,...,1439414,,,,,chr1_1439454_A_G_b38:Stomach:k562,chr1_1439454_A_G_b38:Stomach:k562:ATAD3C,-0.168652,0.159428,False
30727,chr1:1439425:G:C,chr1:1439454:A:G,rare,GTEX-117YW,Stomac,Stomach,0,2.3566,-0.726346,k562,...,1439414,,,,,chr1_1439454_A_G_b38:Stomach:k562,chr1_1439454_A_G_b38:Stomach:k562:ATAD3C,-0.168652,0.159428,False
30728,chr1:1439425:G:C,chr1:1439454:A:G,rare,GTEX-117YX,Stomac,Stomach,0,0.2398,-0.726346,k562,...,1439414,,,,,chr1_1439454_A_G_b38:Stomach:k562,chr1_1439454_A_G_b38:Stomach:k562:ATAD3C,-0.168652,0.159428,False


In [16]:
# define a function for returning a dictionary with everything needed for plotting
def dict4plot_v2 (raw_dataDF):
    dict2return = {}
    # add cell types
    dict2return['K562'] = {}
    dict2return['HEPG2'] = {}
    dict2return['SKNSH'] = {}
    # iterate through the unique tissue : variant : gene pairs from the GTEx data I pulled
    for association in tqdm(raw_dataDF['fullFilter_id'].unique()):
        # break out the association id
        leadVar = association.split(':')[0]
        tissue = association.split(':')[1]
        gene = association.split(':')[-1] 
        varAll = raw_dataDF[(raw_dataDF['gtex_id_leadVar'] == leadVar) & (raw_dataDF['Gene_Name'] == gene) & (raw_dataDF['Tissue_Name'] == tissue)].copy()
        # check the number of cell types predicted in that cell type
        for cell in varAll['cell_type'].unique():
            # filter those for only the lead variant
            varLeadOnly = varAll[(varAll['Variant_Tag'] == 'lead') & (varAll['cell_type'] == cell)].copy()
            # filter those for rare variants
            varRareOnly = varAll[(varAll['Variant_Tag'] == 'rare') & (varAll['cell_type'] == cell)].copy()
            # check if there are rare variants present
            #if len(varRareOnly) > 0:
            # get enhancer id
            enhID = varLeadOnly['enhancer_ids'].unique()[0]
                # combine with gene id to prevent overwrite
            fullID = f'{enhID}_{gene}_{tissue}_{varLeadOnly['gtex_id'].unique()[0]}'
                # update that cell type's dict
            dict2return[cell.upper()][fullID] = {'chrom' : varLeadOnly['gtex_id'].unique()[0].split('_')[0],
                                                    'leadVar' : varLeadOnly['gtex_id'].unique()[0],
                                                    'tissue' : tissue,
                                                    'gene' : gene,
                                                    'leadVar_pos' : int(varLeadOnly['gtex_id'].unique()[0].split('_')[1]),
                                                    'leadVar_pval' : varLeadOnly['lead_pval_nom'].tolist()[0],
                                                    'leadVar_z' : varLeadOnly['z_lead'].tolist()[0],
                                                    'rareVars' : [int(i.split('_')[1]) for i in list(varRareOnly['gtex_id'].unique())],
                                                    'rareVars_z' : varLeadOnly['z_rare'].tolist()[0],
                                                    'enhStart' : varLeadOnly['enhancer_start'].tolist()[0],
                                                    'is_phenocopy' : varLeadOnly['is_phenocopy'].tolist()[0]}
                
            #else:
            #    continue
    return dict2return

In [17]:
dict2plot_v2 = dict4plot_v2(tian_plotted_data)

 11%|█         | 14/129 [00:01<00:10, 11.06it/s]

100%|██████████| 129/129 [00:13<00:00,  9.59it/s]


In [ ]:
EH38E3356274

In [18]:
dict2plot_v2['K562']['EH38E3356274_MRPL35_Artery_Coronary_chr2_86221285_A_G_b38']

{'chrom': 'chr2',
 'leadVar': 'chr2_86221285_A_G_b38',
 'tissue': 'Artery_Coronary',
 'gene': 'MRPL35',
 'leadVar_pos': 86221285,
 'leadVar_pval': '6.261984037730866e-22',
 'leadVar_z': -0.6307272171399407,
 'rareVars': [86221247],
 'rareVars_z': 1.4223389145241605,
 'enhStart': 86221064,
 'is_phenocopy': False}

In [96]:
def plot_specific_enhancers(cell,
                            annotation_dict,
                            collapsedSeqletBED):
    # define highlight color
    if cell == 'K562':
        highlight = '#00A79D'
    elif cell == 'HepG2':
        highlight = '#FBB040'
    elif cell == 'SKNSH':
        highlight = '#ED1C24'
    # iterate through all of that cell type's candidates
    for candidate in annotation_dict[cell].keys():
        # get the enhancer id
        enhancer_id = candidate.split('_')[0]
        # get the lead variant
        leadVarID = annotation_dict[cell][candidate].get('leadVar')
        # get the chromosome
        chrom = annotation_dict[cell][candidate].get('chrom')
        # get the start position
        enhStart = annotation_dict[cell][candidate].get('enhStart')
        # get the tissue
        tissue = annotation_dict[cell][candidate].get('tissue')
        # get the gene
        gene = annotation_dict[cell][candidate].get('gene')
        # get the lead variant position
        LeadVar = annotation_dict[cell][candidate].get('leadVar_pos')
        # get the p value of the lead variant
        leadPVal = annotation_dict[cell][candidate].get('leadVar_pval')
        if leadPVal != 'not_sig' and leadPVal != '':
            #titleCol = 'k'
            leadPVal = float(leadPVal)
        # get the z scores
        zLead = float(annotation_dict[cell][candidate].get('leadVar_z'))
        zRare = float(annotation_dict[cell][candidate].get('rareVars_z'))
        # get rare variants
        otherVars = annotation_dict[cell][candidate].get('rareVars')
        # get phenocopy status
        phenocopy = annotation_dict[cell][candidate].get('is_phenocopy')
        if phenocopy == False:
            titleCol = 'red'
        else:
            titleCol = 'green'
        # load the tensors
        plotLogo_tensors = torch.load(f'/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/seqlets_calling/seqlets_by_chrom/{chrom}_plotLogo_tensors.pt')
        
        # get the seqlets for the BED file
        enhBED = collapsedSeqletBED[collapsedSeqletBED['enhancer_id'] == enhancer_id]
        # pull the tensors for the enhancer for that cell type
        if cell == 'HEPG2':
            tensors2plot = plotLogo_tensors['HepG2'][enhancer_id]
        else:
            tensors2plot = plotLogo_tensors[cell][enhancer_id]
    
        # get the max/min attributions for highlighting seqlets
        minAttrib = tensors2plot.min().min() * 1.05
        maxAttrib = tensors2plot.max().max() * 1.05
    
        # build the DF for plotting with logomaker
        dfFromTensor = pd.DataFrame(tensors2plot.numpy()).T.copy()
        renamedDF = dfFromTensor.rename(columns={0: 'A', 1: 'C', 2: 'G', 3: 'T'})
    
        # remove zero-padded positions at ends
        renamedDF = renamedDF.loc[renamedDF.sum(axis=1) != 0]
    
        # adjust enhStart to account for left padding, then reset index to start at 0
        enhStart = enhStart + renamedDF.index[0]
        renamedDF = renamedDF.reset_index(drop=True)
        seq_length = len(renamedDF)

        # set plot parameters
        matplotlib.rcParams['pdf.fonttype'] = 42
        matplotlib.rcParams['ps.fonttype'] = 42
        matplotlib.rcParams['figure.dpi'] = 300
        
        # generate base plot with extra space for annotations
        fig, ax = plt.subplots(figsize=[20, 2.5])
        logoCheck = logomaker.Logo(renamedDF, ax=ax)
        
        # set y-limits based on data only
        logoCheck.ax.set_ylim(minAttrib, maxAttrib)
        
        # track annotation positions to avoid collisions
        top_annotations = []
        bottom_annotations = []
        min_spacing = 3
        # define helper function for plotting
        def get_annotation_level(x_center, text_half_width, existing_annotations):
                    """Find the lowest level where this annotation won't overlap with others."""
                    level = 0
                    while True:
                        conflict = False
                        for other_center, other_half_width, other_level in existing_annotations:
                            if other_level == level:
                                distance = abs(x_center - other_center)
                                min_dist = text_half_width + other_half_width + min_spacing
                                if distance < min_dist:
                                    conflict = True
                                    break
                        if not conflict:
                            return level
                        level += 1

        # highlight seqlets and collect annotation info
        for idx, row in enhBED.iterrows():
            start, stop, actClass, tf = row[['start', 'end', 'activity_class', 'tf_family']].tolist()
            start = int(start - enhStart)
            stop = int(stop - enhStart - 1)
            x_center = (stop + start) / 2
            x_center_norm = x_center / seq_length  # normalize to 0-1 for axes coords
            tf_label = tf.split('_')[0]
            text_half_width = len(tf_label) * 2.5
            
            if actClass == 'Activator':
                logoCheck.highlight_position_range(pmin=start, pmax=stop, alpha=0.5, color='gainsboro', edgecolor='k', floor=0)
                
                level = get_annotation_level(x_center, text_half_width, top_annotations)
                top_annotations.append((x_center, text_half_width, level))
                
                # annotate in axes coords (outside plot area)
                y_offset = 1.05 + level * 0.05
                ax.annotate(tf_label,
                            xy=(x_center_norm, 1.0),  # anchor at top of axes
                            xycoords='axes fraction',
                            xytext=(x_center_norm, y_offset),
                            textcoords='axes fraction',
                            fontsize=8, fontstyle='italic',
                            ha='center', va='bottom',
                            annotation_clip=False)
            else:
                logoCheck.highlight_position_range(pmin=start, pmax=stop, alpha=0.5, color='gainsboro', edgecolor='k', ceiling=0)
                
                level = get_annotation_level(x_center, text_half_width, bottom_annotations)
                bottom_annotations.append((x_center, text_half_width, level))
                
                # annotate in axes coords (outside plot area)
                y_offset = -0.01 - level * 0.12
                ax.annotate(tf_label,
                            xy=(x_center_norm, 0.0),  # anchor at bottom of axes
                            xycoords='axes fraction',
                            xytext=(x_center_norm, y_offset),
                            textcoords='axes fraction',
                            fontsize=8, fontstyle='italic',
                            ha='center', va='top',
                            annotation_clip=False)
        # highlight lead Variant
        LeadVar = int(LeadVar - enhStart - 1)
        lead_contribution = renamedDF.iloc[LeadVar].sum()
        if lead_contribution >= 0:
            logoCheck.highlight_position(p=LeadVar, color='red', alpha=0.5, floor=0)
        else:
            logoCheck.highlight_position(p=LeadVar, color='red', alpha=0.5, ceiling=0)
        # highlight larger effect variants
        for var in otherVars:
            var2plot = int(var - enhStart - 1)
            var_contribution = renamedDF.iloc[var2plot].sum()
            if var_contribution >= 0:
                logoCheck.highlight_position(p=var2plot, color='deepskyblue', alpha=0.5, floor=0)
            else:
                logoCheck.highlight_position(p=var2plot, color='deepskyblue', alpha=0.5, ceiling=0)
        
        logoCheck.style_spines(visible=False)
        # set x-axis ticks to genomic positions at 50bp intervals
        first_tick_genomic = ((enhStart // 50) + 1) * 50
        tick_positions = []
        tick_labels = []
        
        for genomic_pos in range(first_tick_genomic, enhStart + seq_length, 50):
            plot_pos = genomic_pos - enhStart
            if 0 <= plot_pos < seq_length:
                tick_positions.append(plot_pos)
                tick_labels.append(str(genomic_pos))
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right', rotation_mode='anchor')
        ax.tick_params(axis='x', which='major', length=5, pad=5)

        first_minor_genomic = ((enhStart // 10) + 1) * 10
        first_major_genomic = ((enhStart // 50) + 1) * 50

        major_tick_positions = []
        major_tick_labels = []
        minor_tick_positions = []

        for genomic_pos in range(first_minor_genomic, enhStart + seq_length, 10):
            plot_pos = genomic_pos - enhStart - 1
            if 0 <= plot_pos < seq_length:
                if genomic_pos % 50 == 0:  # major tick
                    major_tick_positions.append(plot_pos)
                    major_tick_labels.append(str(genomic_pos))
                else:  # minor tick
                    minor_tick_positions.append(plot_pos)

        ax.set_xticks(major_tick_positions)
        ax.set_xticklabels(major_tick_labels, rotation=45, ha='right', rotation_mode='anchor')
        ax.tick_params(axis='x', which='major', length=5, pad=5)

        ax.set_xticks(minor_tick_positions, minor=True)
        ax.tick_params(axis='x', which='minor', length=2.5)
        
        max_top_level = max([level for _, _, level in top_annotations], default=-1)
        title_y = 1.10 + (max_top_level + 1) * 0.05
        if leadPVal == 'not_sig' or leadPVal == '':
            ax.set_title(f'{leadVarID} | {tissue} | {gene} | p = {leadPVal} | {enhancer_id} | {cell} | Lead Z: {zLead:.2} | Rare Z: {zRare:.2}', loc='left', y=title_y, color=titleCol, fontweight='bold')
        else:
            ax.set_title(f'{leadVarID} | {tissue} | {gene} | p = {leadPVal:.2e} | {enhancer_id} | {cell} | Lead Z: {zLead:.2} | Rare Z: {zRare:.2}', loc='left', y=title_y, color=titleCol, fontweight='bold')
        
        # adjust subplot to make room for annotations
        plt.subplots_adjust(top=0.75, bottom=0.25)
        plt.tight_layout()
        if phenocopy == True:
            plt.savefig(f'plots_out/phenocopy_true/{cell}/{leadVarID}_{tissue}_{gene}_{enhancer_id}_{cell}.png', dpi=300)
        else:
            plt.savefig(f'plots_out/phenocopy_false/{cell}/{leadVarID}_{tissue}_{gene}_{enhancer_id}_{cell}.png', dpi=300)
        plt.close()

In [97]:
# plot k562 examples
plot_specific_enhancers(
    'K562',
    dict2plot_v2,
    k_mpac_collapsed_dels
)

In [98]:
# plot hepg2 examples
plot_specific_enhancers(
    'HEPG2',
    dict2plot_v2,
    h_mpac_collapsed_dels
)

In [99]:
# plot sknsh examples
plot_specific_enhancers(
    'SKNSH',
    dict2plot_v2,
    s_mpac_collapsed_dels
)